# Temperature Time Series Comparison

Compares near-surface temperature (°C) at a single point from three NWP models served by `tiles.openscicomp.io`:
- **NOAA GFS** analysis (`noaa-gfs-analysis`)
- **NOAA HRRR** analysis (`noaa-hrrr-analysis-v0-2-0`)
- **ECMWF IFS ENS** forecast, ensemble member 0 (`ecmwf-ifs-ens-forecast`)

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor

In [ ]:
BASE_URL = "https://tiles.openscicomp.io"

# Point of interest — Kansas City, MO
LAT, LON = 39.1, -94.6

# Time range: last 7 days
END   = datetime(2026, 3, 10)
START = END - timedelta(days=7)

# Datasets and how to query them
DATASETS = [
    {
        "id": "noaa-gfs-analysis",
        "label": "GFS Analysis",
        "variable": "temperature_2m",
        "extra_params": {},
        "step_hours": 6,
        "time_dim": "time",
    },
    {
        "id": "noaa-hrrr-analysis-v0-2-0",
        "label": "HRRR Analysis",
        "variable": "temperature_2m",
        "extra_params": {},
        "step_hours": 1,
        "time_dim": "time",
    },
    {
        "id": "ecmwf-ifs-ens-forecast",
        "label": "ECMWF ENS (member 0)",
        "variable": "temperature_2m",
        "extra_params": {"lead_time": "nearest::0 hours", "ensemble_member": "nearest::0"},
        "step_hours": 6,
        "time_dim": "init_time",
    },
]

In [ ]:
def fetch_point(dataset_id, variable, lat, lon, time_dim, timestamp, extra_params):
    url = f"{BASE_URL}/datasets/{dataset_id}/tiles/point"
    params = {
        "variables": variable,
        "lat": lat,
        "lon": lon,
        time_dim: f"nearest::{timestamp}",
        **extra_params,
    }
    resp = requests.get(url, params=params, timeout=30)
    data = resp.json()
    return data.get("value")


def fetch_timeseries(ds_config):
    timestamps = []
    t = START
    while t <= END:
        timestamps.append(t)
        t += timedelta(hours=ds_config["step_hours"])

    time_dim = ds_config["time_dim"]

    def fetch_one(t):
        ts = t.strftime("%Y-%m-%dT%H:%M:%S")
        return fetch_point(
            ds_config["id"], ds_config["variable"], LAT, LON,
            time_dim, ts, ds_config["extra_params"]
        )

    with ThreadPoolExecutor(max_workers=8) as ex:
        values = list(ex.map(fetch_one, timestamps))

    return pd.Series(values, index=pd.DatetimeIndex(timestamps), name=ds_config["label"])

In [ ]:
print(f"Fetching temperature at ({LAT}°N, {LON}°E) from {START.date()} to {END.date()}...")
series = {}
for ds in DATASETS:
    print(f"  {ds['label']}...")
    series[ds["label"]] = fetch_timeseries(ds)
print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = {"GFS Analysis": "steelblue", "HRRR Analysis": "darkorange", "ECMWF ENS (member 0)": "seagreen"}
for label, s in series.items():
    ax.plot(s.index, s.values, label=label, color=colors[label], linewidth=1.5)

ax.set_title(f"2-m Temperature at ({LAT}°N, {LON}°E) — Kansas City, MO", fontsize=13)
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Date (UTC)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()